# Lecture 06: Complex Types & Graph Analysis
This notebook guides you through handling nested arrays, key-value maps, structured json documents, data pivoting, and modeling social networks/graphs using nodes and edges.

### 1. Initialize SparkSession
Setting up local master.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lecture06_ComplexTypes") \
    .master("local[2]") \
    .getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark Session initialized successfully!")

### 2. Setup Mock Complex Dataset
We construct user records containing lists of skills and key-value maps of properties.

In [ ]:
complex_data = [
    ("Alice", ["SQL", "Python"], {"level": "Senior", "status": "Active"}),
    ("Bob", ["Scala", "Java", "C++"], {"level": "Junior", "status": "Active"}),
    ("Charlie", ["Go"], {"level": "Mid", "status": "Inactive"})
]
df = spark.createDataFrame(complex_data, ["name", "skills", "properties"])
df.show(truncate=False)
df.printSchema()

### 3. Creating Array Columns
Gathering multiple separate columns into a single ArrayType column using `array()`.

In [ ]:
from pyspark.sql.functions import array, col

spark.range(1).select(
    array(col("id"), col("id") * 2, col("id") * 3).alias("multiples_arr")
).show()

### 4. Exploding Array Elements (explode)
Exploding array elements into separate rows. This flattens the dataset structure.

In [ ]:
from pyspark.sql.functions import explode

df.select("name", explode("skills").alias("skill")).show()

### 5. Positional Exploding (posexplode)
Similar to explode(), but returns the position index of the element alongside the values.

In [ ]:
from pyspark.sql.functions import posexplode

df.select("name", posexplode("skills")).show()

### 6. Checking Array Elements inclusion
Using `array_contains()` to check if a specific element is present in the array.

In [ ]:
from pyspark.sql.functions import array_contains

df.select("name", array_contains("skills", "Python").alias("knows_python")).show()

### 7. Sorting Array elements
Sorting array elements alphabetically or numerically using `array_sort()`.

In [ ]:
from pyspark.sql.functions import array_sort

df.select("name", array_sort("skills").alias("sorted_skills")).show(truncate=False)

### 8. Creating Key-Value Map columns
Generating a MapType column from key-value pairs.

In [ ]:
from pyspark.sql.functions import create_map, lit

map_df = spark.range(1).select(
    create_map(lit("Key_A"), lit(100), lit("Key_B"), lit(200)).alias("custom_map")
)
map_df.show(truncate=False)

### 9. Map Keys Extraction
Extracting all keys from a map column using `map_keys()`.

In [ ]:
from pyspark.sql.functions import map_keys

df.select("name", map_keys("properties").alias("keys")).show(truncate=False)

### 10. Map Values Extraction
Extracting all values from a map column using `map_values()`.

In [ ]:
from pyspark.sql.functions import map_values

df.select("name", map_values("properties").alias("vals")).show(truncate=False)

### 11. Creating Struct columns
Nesting columns inside a StructType column using `struct()` to group attributes.

In [ ]:
from pyspark.sql.functions import struct

struct_df = df.select("name", struct(col("skills"), col("properties")).alias("user_details"))
struct_df.show(truncate=False)
struct_df.printSchema()

### 12. Parsing JSON Strings (from_json)
Parsing JSON string columns into structured objects using predefined schemas.

In [ ]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

json_df = spark.createDataFrame([("101", '{"product": "Phone", "quantity": 2}')], ["id", "json_str"])
json_schema = StructType([
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True)
])

parsed_json = json_df.withColumn("order", from_json(col("json_str"), json_schema))
parsed_json.show(truncate=False)
parsed_json.printSchema()

### 13. Serializing Structs to JSON (to_json)
Converting structured objects back into JSON string representations.

In [ ]:
from pyspark.sql.functions import to_json

parsed_json.select("id", to_json("order").alias("order_json_str")).show(truncate=False)

### 14. Path-based JSON extraction
Extracting values from JSON columns using path expressions with `get_json_object()`.

In [ ]:
from pyspark.sql.functions import get_json_object

json_df.select("id", get_json_object("json_str", "$.product").alias("product_name")).show()

### 15. Pivot Table transformation
Rotating unique row values into columns to summarize datasets.

In [ ]:
pivot_df = spark.createDataFrame([
    ("A", "x", 10), ("A", "y", 20),
    ("B", "x", 30), ("B", "y", 40)
], ["user", "metric", "val"])

pivoted = pivot_df.groupBy("user").pivot("metric").sum("val")
pivoted.show()

### 16. Unpivot Table (stack)
Transposing pivoted columns back into rows using the SQL stack expression.

In [ ]:
from pyspark.sql.functions import expr

pivoted.select(
    "user",
    expr("stack(2, 'x', x, 'y', y) as (metric, val)")
).show()

### 17. Graph representation: Vertices
Modeling vertices representing nodes in a graph.

In [ ]:
vertices = spark.createDataFrame([
    ("U1", "Alice"),
    ("U2", "Bob"),
    ("U3", "Charlie")
], ["id", "name"])
vertices.show()

### 18. Graph representation: Edges
Modeling edges representing direct relationships between nodes.

In [ ]:
edges = spark.createDataFrame([
    ("U1", "U2"),
    ("U2", "U3"),
    ("U1", "U3"),
    ("U3", "U1")
], ["src", "dst"])
edges.show()

### 19. Graph: In-Degree Calculation
Counting the number of incoming edges for each node.

In [ ]:
in_degree = edges.groupBy("dst").count().withColumnRenamed("count", "in_degree")
in_degree.show()

### 20. Graph: Out-Degree Calculation
Counting the number of outgoing edges for each node.

In [ ]:
out_degree = edges.groupBy("src").count().withColumnRenamed("count", "out_degree")
out_degree.show()

### 21. Stop Spark Session
Stopping resources cleanly.

In [ ]:
spark.stop()